In [0]:
control_table = spark.read.table('data.late_dim.control_table')
import pyspark.sql.functions as f

timestamp


In [0]:
import pytz
from datetime import datetime
fact_last_time = control_table.filter((f.col('status')== 'SUCCESS') & (f.col('pipeline_name') == 'fact_sales'))\
    .select(f.max('last_ingestion_time'))\
    .collect()[0][0]
if fact_last_time is None :
    last_run_fact = '2026-04-01 00:00:00'
else:
    last_run_fact = fact_last_time.strftime("%Y-%m-%d %H:%M:%S")

dim_cust_last_time = control_table.filter((f.col('status')== 'SUCCESS') & (f.col('pipeline_name') == 'dim_customer'))\
    .select(f.max('last_ingestion_time'))\
    .collect()[0][0]
if dim_cust_last_time is None :
    last_run_cust = '2026-04-01 00:00:00'
else:
    last_run_cust = dim_cust_last_time.strftime("%Y-%m-%d %H:%M:%S")

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).strftime("%Y-%m-%d %H:%M:%S")

insert and update customer data

In [0]:
try:
  # update dummy records
  spark.sql(f'''
      merge into data.late_dim.dim_customer t
      using data.late_dim.src_customer s
      on s.customer_id = t.customer_id
      when matched and t.customer_name= 'unknown' and t.contact is null and t.email is null 
      then update set 
        t.customer_name = s.customer_name,
        t.customer_city = s.customer_city,
        t.email = s.email,
        t.contact = s.contact,
        t.start_date = s.created_date,
        t.is_active = True,
        t.ingestion_time = current_timestamp()
        ''')

  # expire old records if updated happend after last run
  spark.sql(f'''
      merge into data.late_dim.dim_customer t
      using data.late_dim.src_customer s
      on s.customer_id = t.customer_id and t.is_active = True
      when matched 
      and (s.contact != t.contact 
            or s.email != t.email 
            or s.customer_city != t.customer_city
            or s.customer_name != t.customer_name)
      then update set
        t.is_active = False,
        t.end_date = s.updated_date
          ''')
    
  #insert new records
  spark.sql(f'''
      INSERT INTO data.late_dim.dim_customer (
      customer_id, customer_name, customer_city, email, contact,start_date, end_date, is_active, ingestion_time)
    SELECT 
      s.customer_id, s.customer_name, s.customer_city, s.email, s.contact,s.created_date,NULL, true, current_timestamp()
    FROM data.late_dim.src_customer s
    LEFT JOIN data.late_dim.dim_customer t
      ON s.customer_id = t.customer_id AND t.is_active = true
    WHERE 
      t.customer_id IS NULL
      OR (
        s.contact <> t.contact OR
        s.email <> t.email OR
        s.customer_city <> t.customer_city OR
        s.customer_name <> t.customer_name )
        ''')

  # source count 
  cust_src_count = spark.sql(f'''
      select count(customer_id) from data.late_dim.src_customer
      where created_date >TIMESTAMP('{last_run_cust}') or updated_date > TIMESTAMP('{last_run_cust}')
      ''').collect()[0][0]
    
  # update count
  update_count = spark.sql(f'''
      select count(cust_sk) from data.late_dim.dim_customer
      where end_date > TIMESTAMP('{last_run_cust}')
      ''').collect()[0][0]
    
  # insert count
  insert_count = spark.sql(f'''
      select count(cust_sk) from data.late_dim.dim_customer
      where start_date > TIMESTAMP('{last_run_cust}') and is_active = true
      ''').collect()[0][0] - update_count

  # update the control table
  spark.sql(f'''
       insert into data.late_dim.control_table
      (pipeline_name,last_ingestion_time,status,src_count,insert_count,update_count,error_msg)
      values
      ('dim_customer',TIMESTAMP('{current_time}'),'SUCCESS',{cust_src_count},{insert_count},{update_count},null)
      ''')

except Exception as e:
  error_msg = str(e).replace("'", " ")
  spark.sql(f'''
      insert into data.late_dim.control_table
      (pipeline_name,last_ingestion_time,status,src_count,insert_count,update_count,error_msg) 
      values
      ('dim_customer',TIMESTAMP('{current_time}'),'FAILED', 0, 0,0,'{error_msg}')
                            ''')

insert dummy row when dimentions are not available and insert sales data 

In [0]:
try:
    # insert dummy row when dimentions are not available
    spark.sql(f'''
        merge into data.late_dim.dim_customer t
        using (select distinct customer_id from data.late_dim.src_sales where sale_date > '{last_run_fact}') s
        on s.customer_id = t.customer_id
        when not matched then 
        insert (customer_id, customer_name, customer_city, email, contact, start_date,end_date,is_active,ingestion_time)
        values (s.customer_id, 'unknown','unknown', null, null, '2000-01-01',null,True,current_timestamp())
        ''')
    
    # insert fact data
    spark.sql(f'''
        insert into data.late_dim.fact_sales(sale_id,cust_sk,sale_date,sale_amount,ingestion_time)
        select s.sale_id,c.cust_sk,s.sale_date,s.sale_amount,current_timestamp()
        from (select * from data.late_dim.src_sales where sale_date > TIMESTAMP('{last_run_fact}')) s
        join data.late_dim.dim_customer c
        on s.customer_id = c.customer_id
        and s.sale_date >= c.start_date
        and (s.sale_date < c.end_date or c.end_date is null)
        ''')
    
    sale_src_count = spark.sql(f'''
        select count(sale_id) from data.late_dim.fact_sales where sale_date > TIMESTAMP('{last_run_fact}')
        ''').collect()[0][0]
    
    sale_des_count = spark.sql(f'''
        select count(sale_id) from data.late_dim.fact_sales where sale_date > TIMESTAMP('{last_run_fact}')
        ''').collect()[0][0]

    spark.sql(f'''
        insert into data.late_dim.control_table
        (pipeline_name,last_ingestion_time,status,src_count,insert_count,update_count,error_msg) 
        values
        ('fact_sales',TIMESTAMP('{current_time}'),'SUCCESS', '{sale_src_count}','{sale_des_count}',0,Null)
        ''')

except Exception as e:
    error_msg = str(e).replace("'", " ")
    spark.sql(f'''
        insert into data.late_dim.control_table
        (pipeline_name,last_ingestion_time,status,src_count,insert_count,update_count,error_msg) 
        values
        ('fact_sales',TIMESTAMP('{current_time}'),'FAILED', 0, 0,0,'{error_msg}')
                              ''')